# **자연어 처리 (NLP)**

## **NLP 감정분석 (Pytorch)**
RNN을 활용하여 텍스트의 감성을 분석하는 모델

입력된 단어들의 순차적인 정보를 파악하여 긍정(1) 또는 부정(0)을 예측하는 과정


### **코드 구조**
Embedding층: 단어 인덱스를 고차원 벡터로 변환

RNN층 (은닉층): 16개의 Hidden Unit으로 문맥 정보 처리

출력층 (Linear): 최종 예측값 1개 생성

### **활성화 함수**
sigmoid

### **손실 함수**
BCELoss

### **Optimizer**
Adam

In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np

In [15]:
sentences = ['happy good', 'bad sad', 'happy happy', 'bad bad']
labels = [1, 0, 1, 0]

w_li = " ".join(sentences).split()
w_li = list(set(w_li))
w_di = {w: i for i, w in enumerate(w_li)}
vocab_size = len(w_di)

print(f'단어 사전: {w_di}')

단어 사전: {'good': 0, 'happy': 1, 'bad': 2, 'sad': 3}


In [16]:
class SimpleNLPDataset(Dataset):
  def __init__(self, sentences, labels, w_di):
    self.data = []
    for s in sentences:
      self.data.append([w_di[w] for w in s.split()])
    self.labels = labels

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return torch.LongTensor(self.data[idx]), torch.FloatTensor([self.labels[idx]])
dataset = SimpleNLPDataset(sentences, labels, w_di)
loader = DataLoader(dataset, batch_size = 2, shuffle = True)

In [17]:
class SimpleRNN(nn.Module):
  def __init__ (self, vocab_size, embedding_dim, hidden_size) :
    super(SimpleRNN, self).__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim)
    self.rnn = nn.RNN(embedding_dim, hidden_size, batch_first = True)
    self.fc = nn.Linear(hidden_size, 1)
    self.sigmoid = nn.Sigmoid()

  def forward(self, x) :
    x = self.embedding(x)
    out, hidden = self.rnn(x)
    return self.sigmoid(self.fc(hidden.squeeze(0)))

model = SimpleRNN(vocab_size, embedding_dim = 8, hidden_size = 16)

In [18]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

In [19]:
for epoch in range(100):
  for inputs, targets in loader:
    optimizer.zero_grad()
    outputs = model(inputs)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()
  print(f'''
    Epochs {epoch}
    예측값 {outputs}
    Loss {loss.item():.4f}
    --------------''')


    Epochs 0
    예측값 tensor([[0.4536],
        [0.4522]], grad_fn=<SigmoidBackward0>)
    Loss 0.6990
    --------------

    Epochs 1
    예측값 tensor([[0.4339],
        [0.5220]], grad_fn=<SigmoidBackward0>)
    Loss 0.6095
    --------------

    Epochs 2
    예측값 tensor([[0.6177],
        [0.5757]], grad_fn=<SigmoidBackward0>)
    Loss 0.5170
    --------------

    Epochs 3
    예측값 tensor([[0.6456],
        [0.3085]], grad_fn=<SigmoidBackward0>)
    Loss 0.4033
    --------------

    Epochs 4
    예측값 tensor([[0.2699],
        [0.2558]], grad_fn=<SigmoidBackward0>)
    Loss 0.3050
    --------------

    Epochs 5
    예측값 tensor([[0.2144],
        [0.8161]], grad_fn=<SigmoidBackward0>)
    Loss 0.2222
    --------------

    Epochs 6
    예측값 tensor([[0.1586],
        [0.1344]], grad_fn=<SigmoidBackward0>)
    Loss 0.1585
    --------------

    Epochs 7
    예측값 tensor([[0.9038],
        [0.1091]], grad_fn=<SigmoidBackward0>)
    Loss 0.1083
    --------------

    Epochs 8
    예측값 te

In [29]:
model.eval()
with torch.no_grad():
  test_input = torch.LongTensor([[w_di['happy'], w_di['good']]])
  prediction = model(test_input)
  print(f'예측 결과: {prediction.item():.4f}')

예측 결과: 0.9989


In [30]:
model.eval()
with torch.no_grad():
  test_input = torch.LongTensor([[w_di['sad'], w_di['bad']]])
  prediction = model(test_input)
  print(f'예측 결과: {prediction.item():.4f}')

예측 결과: 0.9945
